In [40]:
# to import things from script folder on this notebooks
import sys; sys.path.append('..')
from Bio import AlignIO, SeqIO
import pandas as pd
from pathlib import Path
import itertools as it
import subprocess
from scipy.stats import entropy
from collections import Counter

from src.conf import DATA_RAW, DATA_INTERIM, DATA_PROCESSED, protein_names
from src._aacon import AAConCommandline



In [54]:
def shannon_entropy(dna_sequence):
    bases = Counter([tmp_base for tmp_base in dna_sequence])
    # define distribution
    dist = [x/sum(bases.values()) for x in bases.values()]
    # use scipy to calculate entropy
    entropy_value = entropy(dist, base=2)
 
    return entropy_value

In [75]:
def get_shannon(alingment):
    
    col_values = []
    for i in range(ali.get_alignment_length()):
        col = ali[:,i]
        s = round(shannon_entropy(col),5)
        col_values.append(s)
    
    return col_values
    


In [71]:
mafft_folder = DATA_PROCESSED/"mafftout"
aacon_out = DATA_PROCESSED/'aacon'
aacon_cmd = "/mnt/d/Bioinformatics/software/compbio-conservation-1.1.jar"

if not aacon_out.is_dir():
    aacon_out.mkdir()

In [94]:
c = 'China'
mafft_country = mafft_folder.glob(f'*{c}.ali')


for mfile in mafft_country:
    # parse ali for calculate shannon entropy
    ali = AlignIO.read(mfile,'fasta')
    ali_entropy = get_shannon(ali)
    # protein name
    pname = mfile.name[:-4].split('_')[0]
    
    # Run aacon 
    outaacon = aacon_out/f'{pname}_aacon_{c}.txt'
    aacon_cline = AAConCommandline(cmd=aacon_cmd,i=str(mfile),o=str(outaacon),n=True)
    cmd = 'java -jar '+ str(aacon_cline)
    print(cmd)
    subprocess.run(cmd,shell=True, check=True)
    
    
    # write shannon entropy to same aacon outfile
    shannon_line = '#SHANNON '+' '.join([str(x) for x in ali_entropy])
    with open(outaacon,'r+') as f:
        # remove last new line
        lines = f.readlines()[:-1]
        f.seek(0)
        f.writelines(lines)
        f.write(shannon_line)
        f.truncate()
        


java -jar /mnt/d/Bioinformatics/software/compbio-conservation-1.1.jar -i=/mnt/d/Python_projects/sars-cov-2_proteome/data/processed/mafftout/E_mafft_China.ali -o=/mnt/d/Python_projects/sars-cov-2_proteome/data/processed/aacon/E_aacon_China.txt -n
java -jar /mnt/d/Bioinformatics/software/compbio-conservation-1.1.jar -i=/mnt/d/Python_projects/sars-cov-2_proteome/data/processed/mafftout/M_mafft_China.ali -o=/mnt/d/Python_projects/sars-cov-2_proteome/data/processed/aacon/M_aacon_China.txt -n
java -jar /mnt/d/Bioinformatics/software/compbio-conservation-1.1.jar -i=/mnt/d/Python_projects/sars-cov-2_proteome/data/processed/mafftout/NS3_mafft_China.ali -o=/mnt/d/Python_projects/sars-cov-2_proteome/data/processed/aacon/NS3_aacon_China.txt -n
java -jar /mnt/d/Bioinformatics/software/compbio-conservation-1.1.jar -i=/mnt/d/Python_projects/sars-cov-2_proteome/data/processed/mafftout/NS6_mafft_China.ali -o=/mnt/d/Python_projects/sars-cov-2_proteome/data/processed/aacon/NS6_aacon_China.txt -n
java -ja